# Feature Engineering IV

**Downsampling Return**

Author: Pete King

This notebook is a fresh take on feature engineering, where we treat the object of our study (return) as a random variable (**R**) and investigate its properties over different time horizons.  We use the Pandas 'resample' method to examine how return behaves in weekly, bi-weekly, and monthly frequencies.

Our hope is that looking at how return changes over longer periods will even out some of the randomness we saw in the daily values and allow us to make better long-term predictions.

The key difference with this approach compared to our earlier approach is that our models will no longer make a prediction at each daily timestamp.  For example, if we downsample to a monthly periodicity, our models will make 12 predictions in a year.


In [1]:
#123456789012345678901234567890123456789012345678901234567890123456789012345678
import json

import numpy as np
import pandas as pd

import data_prep as dp

ETF_DATA_FILE = 'etf_raw_data.csv'
INDICIES_DATA_FILE = 'filled_indicies_data.csv'
DOWNSAMPLE_FOLDER = 'data/downsample/'
OUTLOOK = 21  # No. business days in the volatility forecast

## I. Import data

In [2]:
etf_df = pd.read_csv(
    ETF_DATA_FILE,
    index_col='date',
    parse_dates=True
)
ind_df = pd.read_csv(
    INDICIES_DATA_FILE,
    index_col='date',
    parse_dates=True
)

In [3]:
etf_df

,BIL,BND,GLD,HYG,IEF,IWM,LQD,QQQ,SPY,TIP,...,XLB,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
date,,,,,,,,,,,,,,,,,,,,,
1993-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.175379,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.347328,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.398905,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.656818,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.759996,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-27,91.366028,72.861801,414.700012,78.340057,94.285789,243.100006,107.179611,562.580017,634.090027,109.669998,...,48.910000,62.560001,47.810001,159.199997,129.919998,81.779999,40.009998,45.590000,143.259995,105.680000
2026-03-30,91.375999,73.230545,414.579987,78.429619,94.953560,239.610001,107.866783,558.280029,631.969971,110.349998,...,49.090000,61.959999,48.360001,156.610001,127.500000,81.879997,40.209999,45.919998,143.820007,105.660004
2026-03-31,91.375999,73.389999,430.290009,79.175995,95.123001,248.000000,108.543999,577.179993,650.340027,110.360001,...,49.970001,61.259998,49.369999,161.729996,132.899994,81.980003,40.830002,45.889999,146.610001,108.980003


As before, we want to compute return and volatility labels for all ETFs and selected indicies.

We do not apply these transformations to the volatility indexes (VIX, VXN, and MOVE), as they are based on economic models that forecast volatility itself over a roughly 30 calendar day horizon-- they are not price indicies.

In [4]:
# From Tom's notebook:  We also compute return and volatility for these
indicies = [
    # Treasury bond index
    'SHY',
    # Commodity futures
    'CL=F', 'GC=F', 'HG=F', 'NG=F',
    # Equity indicies
    '^FTSE', '^GSPC', '^HSI',
]
# Note that we keep the volatility indicies separate
vol_indicies = ['^VIX', '^VXN', '^MOVE']

# We also make a list of tickers for convenience later
tickers = {}
tickers['ETF'] = list(etf_df.columns)
tickers['other_index'] = indicies
tickers['vol_index'] = vol_indicies

with open('tickers.json', 'w', encoding='utf-8') as f:
    json.dump(tickers, f)

## II. Downsample and transform ETFs

Return represents a change in the price of an asset.  Daily return is given by the formula:

***r_t = log(price_t / price_t-1)***, where r_t represents return, computed as the log of the ratio of closing price on day t to closing price on day (t - 1).

We will base our volatility computations on daily return.  We use the zero-mean assumption for expected return (**E\[R\]**)  Volatility for a period of time is then given by:

***volatility_p = sqrt( sum(r_i^2)/n )***, for i = 1, 2, ..., n where there are n timestamped observations of daily return in the period, and each r_i is a daily return value.


### Compute daily return

In [5]:
# Compute daily return
etf_ret = {}
for ticker in tickers['ETF']:
    # Compute daily return (feature)
    etf_ret[ticker + '_ret'] = np.log(
        etf_df[ticker].values / etf_df[ticker].shift(1).values
    )
etf_ret = pd.DataFrame(etf_ret, index=etf_df.index)
etf_ret.head()

,BIL_ret,BND_ret,GLD_ret,HYG_ret,IEF_ret,IWM_ret,LQD_ret,QQQ_ret,SPY_ret,TIP_ret,...,XLB_ret,XLE_ret,XLF_ret,XLI_ret,XLK_ret,XLP_ret,XLRE_ret,XLU_ret,XLV_ret,XLY_ret
date,,,,,,,,,,,,,,,,,,,,,
1993-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.007087,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.002116,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.010515,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.004176,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Downsample and compute return/volatility features and label

We generate two features:
 - return (past week)
 - historical volatility (past week)

And one label:
 - historical volatility (next 4 weeks)

We forecast out 21 business days (approx. one month) so that we can compare our models with the [VIX](https://www.investopedia.com/terms/v/vix.asp) as a baseline.

In [6]:
freqs = ['W', '2W', 'BME']
etf_rets, etf_vols, etf_targets = {}, {}, {}
for freq in freqs:
    # Compute weekly returns based on daily closing price from etf_df
    etf_rets[freq] = etf_df.resample(
        'W', closed='right', label='right'
    ).pipe(lambda x: np.log(x.last() / x.first())).rename(
        columns=lambda x: f'{x}_ret-h'
    )
    # Compute weekly volatility based on computed daily return from etf_ret
    etf_vols[freq] = etf_ret.resample(
        'W', closed='right', label='right'
    ).apply(dp.volatility).rename(
        columns=lambda x: f'{x[:-4]}_vol-h'
    )
    # Compute volatility target (OUTLOOK) based on daily return from etf_ret
    etf_targets[freq] = etf_ret.shift(-OUTLOOK).rolling(
        OUTLOOK, closed='right'
    ).apply(dp.volatility, raw=True).resample(
        'W', closed='right', label='right'
    ).pipe(lambda x: x.last()).rename(
        columns=lambda x: f'{x[:-4]}_vol_target'
    )

In [7]:
etf_rets[freqs[0]].head()

,BIL_ret-h,BND_ret-h,GLD_ret-h,HYG_ret-h,IEF_ret-h,IWM_ret-h,LQD_ret-h,QQQ_ret-h,SPY_ret-h,TIP_ret-h,...,XLB_ret-h,XLE_ret-h,XLF_ret-h,XLI_ret-h,XLK_ret-h,XLP_ret-h,XLRE_ret-h,XLU_ret-h,XLV_ret-h,XLY_ret-h
date,,,,,,,,,,,,,,,,,,,,,
1993-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.016112,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.008374,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.002153,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.015603,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
etf_vols[freqs[0]].head()

,BIL_vol-h,BND_vol-h,GLD_vol-h,HYG_vol-h,IEF_vol-h,IWM_vol-h,LQD_vol-h,QQQ_vol-h,SPY_vol-h,TIP_vol-h,...,XLB_vol-h,XLE_vol-h,XLF_vol-h,XLI_vol-h,XLK_vol-h,XLP_vol-h,XLRE_vol-h,XLU_vol-h,XLV_vol-h,XLY_vol-h
date,,,,,,,,,,,,,,,,,,,,,
1993-01-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1993-02-07,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.006053,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1993-02-14,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.005165,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1993-02-21,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.012911,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1993-02-28,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.006058,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
etf_targets[freqs[0]].head()

,BIL_vol_target,BND_vol_target,GLD_vol_target,HYG_vol_target,IEF_vol_target,IWM_vol_target,LQD_vol_target,QQQ_vol_target,SPY_vol_target,TIP_vol_target,...,XLB_vol_target,XLE_vol_target,XLF_vol_target,XLI_vol_target,XLK_vol_target,XLP_vol_target,XLRE_vol_target,XLU_vol_target,XLV_vol_target,XLY_vol_target
date,,,,,,,,,,,,,,,,,,,,,
1993-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
# For each frequency, create a combined DataFrame and save as CSV file
for freq in freqs:
    pd.concat(
        [etf_rets[freq], etf_vols[freq], etf_targets[freq]],
        axis=1
    ).to_csv(
        f'{DOWNSAMPLE_FOLDER}{freq}/etf_downsample.csv', index_label='date'
    )

## III. Downsample and transform non-volatility indicies

We apply the same downsampling and transformations (return and volatility computations) to the non-volatility indicies.


### Compute daily return

In [11]:
# Compute daily return
ind_ret = {}
for ticker in indicies:
    # Compute daily return (feature)
    ind_ret[ticker + '_ret'] = np.log(
        ind_df[ticker].values / ind_df[ticker].shift(1).values
    )
ind_ret = pd.DataFrame(ind_ret, index=ind_df.index)
ind_ret.head()

,SHY_ret,CL=F_ret,GC=F_ret,HG=F_ret,NG=F_ret,^FTSE_ret,^GSPC_ret,^HSI_ret
date,,,,,,,,
2002-11-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2002-11-13,0.000122,-0.027796,-0.018035,-0.004880,0.001291,-0.013729,-0.000476,0.000289
2002-11-14,-0.002443,0.003962,-0.002513,-0.007013,-0.002066,0.005865,0.024335,0.012834
2002-11-15,0.000489,0.008661,0.008769,0.009107,0.028537,0.009454,0.006130,0.012732
2002-11-18,0.000000,0.045967,-0.004688,0.013851,0.068440,0.005946,-0.010463,0.005503


### Downsample and compute periodic return, volatility

In [12]:
ind_rets, ind_vols = {}, {}
for freq in freqs:
    # Compute returns based on daily closing price from ind_df
    ind_rets[freq] = ind_df[indicies].resample(
        freq, closed='right', label='right'
    ).pipe(lambda x: np.log(x.last() / x.first())).rename(
        columns=lambda x: f'{x}_ret-h'
    )
    # Compute volatility based on computed daily return from ind_ret
    ind_vols[freq] = ind_ret.resample(
        freq, closed='right', label='right'
    ).apply(dp.volatility).rename(
        columns=lambda x: f'{x[:-4]}_vol-h'
    )

In [13]:
ind_vols[freqs[0]]

,SHY_vol-h,CL=F_vol-h,GC=F_vol-h,HG=F_vol-h,NG=F_vol-h,^FTSE_vol-h,^GSPC_vol-h,^HSI_vol-h
date,,,,,,,,
2002-11-17,0.001247,0.014691,0.010106,0.006243,0.014320,0.008835,0.012550,0.009040
2002-11-24,0.000740,0.026339,0.005417,0.009868,0.033592,0.010933,0.013841,0.004461
2002-12-01,0.001165,0.014847,0.004291,0.013655,0.015589,0.012244,0.015626,0.011109
2002-12-08,0.000683,0.014661,0.006193,0.008183,0.017238,0.009618,0.009181,0.012008
2002-12-15,0.000543,0.016253,0.010026,0.009322,0.047832,0.013271,0.013356,0.007139
...,...,...,...,...,...,...,...,...
2026-03-08,0.000926,0.071739,0.019044,0.015987,0.041552,0.016454,0.008492,0.016179
2026-03-15,0.000977,0.076740,0.015687,0.013001,0.035405,0.008148,0.008319,0.012638
2026-03-22,0.001386,0.029204,0.029673,0.020005,0.024161,0.013885,0.010352,0.012141


### First difference for Treasury Yield

Follow Tom's guidance for transforming Treasury yield using a first difference (i.e., "delta") between original values.

In [14]:
t_yield = {}
for freq in freqs:
    t_yield[freq] = ind_df['^TNX'].resample(
        freq, closed='right', label='right', 
    ).pipe(lambda x: x.last() - x.first())
    t_yield[freq].name = '^TNX_diff'

## IV. Volatility indicies resample

For the volatility indexes, each daily observation is a forecast of annualized volatility for the next 30 days.  When we downsample from daily values to a lower frequency, we simply take the latest (most recent) of daily predictions in the period, as this would be the most recent forecast for annualized volatility over the next 30 days.

In [15]:
vols = {}
for freq in freqs:
    vols[freq] = ind_df[vol_indicies].resample(
        freq, closed='right', label='right', 
    ).pipe(lambda x: x.last())

In [16]:
vols[freqs[0]]

,^VIX,^VXN,^MOVE
date,,,
2002-11-17,26.650000,49.680000,132.350006
2002-11-24,23.160000,46.490002,126.269997
2002-12-01,27.500000,49.480000,127.519997
2002-12-08,28.879999,52.279999,128.460007
2002-12-15,28.180000,50.919998,116.209999
...,...,...,...
2026-03-08,29.490000,31.440001,81.260002
2026-03-15,27.190001,29.870001,91.169998
2026-03-22,26.780001,29.200001,108.839996


In [17]:
# For each frequency, create a combined DataFrame and save as CSV file
for freq in freqs:
    pd.concat(
        [ind_rets[freq], ind_vols[freq], vols[freq], t_yield[freq]],
        axis=1, join='outer', sort=True
    ).to_csv(
        f'{DOWNSAMPLE_FOLDER}{freq}/indicies_downsample.csv',
        index_label='date'
    )